# Chapter 06: Classification fine-tuning mechanics


Classification reuses token representations but changes how they are
pooled and which parameters are updated. These exercises make padding,
parameter freezing, and sequence-position selection explicit.


    This notebook is an independently written practice companion. It
    does not reproduce the book's prose, figures, data, or solutions.
    Read the [Chinese chapter](https://github.com/skindhu/Build-A-Large-Language-Model-CN/blob/main/cn-Book/6.%E7%94%A8%E4%BA%8E%E5%88%86%E7%B1%BB%E4%BB%BB%E5%8A%A1%E7%9A%84%E5%BE%AE%E8%B0%83.md) for the complete narrative
    and exercise statements, and use the
    [canonical MIT companion code](https://github.com/rasbt/LLMs-from-scratch)
    as an additional reference.

    **Prerequisites:** Python 3.10+ and PyTorch 2.2+. Every example is
    synthetic, deterministic, CPU-friendly, and makes no network call.

    **Workflow**

    1. Read the concept and trace the shapes by hand.
    2. Replace one `LLM_TODO` and its `NotImplementedError`.
    3. Run that exercise's check cell.
    4. Explain the result before moving to the next exercise.
    5. Compare with `solution.py` only after making a serious attempt.

In [ ]:
from __future__ import annotations

from collections.abc import Sequence
from typing import Literal

import torch
from torch import Tensor, nn

## Exercise 6.1: Pad sequences and preserve validity

    **Objective.** Right-pad variable-length token rows and return a Boolean attention mask.


Dense batches require a common sequence length. Right-padding places
real tokens first and fills the remainder with `pad_id`. A mask records
which positions are valid so attention, pooling, and loss logic can
ignore padding.

Padding to the longest row in a batch minimizes wasted work; padding to
a fixed maximum can simplify shapes but may add substantial computation.
The mask, not the numeric pad ID alone, is the reliable semantic signal.


    ### Data-flow walkthrough


For rows `[1,2,3]` and `[4]`, automatic length is three. Padded tokens
are `[[1,2,3],[4,pad,pad]]`; the Boolean mask is
`[[T,T,T],[T,F,F]]`. An explicit `max_length` may truncate longer rows
and pad shorter ones.


    ### Hints and common mistakes


Return tensors with identical `[B,T]` shape. Decide truncation before
padding, validate positive explicit lengths, and use Boolean mask dtype.
Do not infer validity later from hidden-state values.

In [ ]:
def pad_and_mask(
    sequences: Sequence[Sequence[int]],
    pad_id: int,
    max_length: int | None = None,
) -> tuple[Tensor, Tensor]:
    # LLM_TODO_CH06_01_PADDING
    raise NotImplementedError("right-pad and return a validity mask")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
tokens, mask = pad_and_mask([[1, 2, 3], [4]], pad_id=0)
assert tokens.tolist() == [[1, 2, 3], [4, 0, 0]]
assert mask.dtype == torch.bool
assert mask.tolist() == [[True, True, True], [True, False, False]]
print("Exercise 6.1 passed")

## Exercise 6.2: Select trainable parameters

    **Objective.** Toggle between classifier-only and full-model fine-tuning and count parameters.


Freezing a parameter sets `requires_grad=False`, preventing gradient
storage and optimizer updates for that parameter. Head-only training
adapts a small classifier while preserving the backbone; full
fine-tuning exposes every parameter to updates.

A configuration helper should first set every parameter consistently,
then enable only the named head when requested. Returning the trainable
parameter count makes the policy observable and testable.


    ### Data-flow walkthrough


In a toy module with `backbone` and `classifier`, head mode should leave
only classifier weight and bias trainable. Full mode should make the
count equal the model's total parameter count.


    ### Hints and common mistakes


Iterate over parameters rather than modules because a module may own
multiple tensors. Validate the head name and mode. Changing
`requires_grad` does not automatically rebuild an already-created
optimizer.

In [ ]:
def configure_trainable(
    model: nn.Module,
    mode: Literal["head", "full"],
    head_name: str = "classifier",
) -> int:
    # LLM_TODO_CH06_02_UNFREEZE
    raise NotImplementedError(
        "set requires_grad according to the selected policy"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
toy_model = nn.ModuleDict({
    "backbone": nn.Linear(4, 4),
    "classifier": nn.Linear(4, 2),
})
head_count = configure_trainable(toy_model, "head")
expected_head = sum(p.numel() for p in toy_model["classifier"].parameters())
assert head_count == expected_head
full_count = configure_trainable(toy_model, "full")
assert full_count == sum(p.numel() for p in toy_model.parameters())
print("Exercise 6.2 passed")

## Exercise 6.3: Choose a sequence representation

    **Objective.** Select either position zero or each row's last valid hidden state.


A decoder produces hidden states `[B,T,D]`. A classifier needs one
vector per sequence, so a policy must choose a position. Position zero
is easy to index but in causal attention it has seen only itself. The
last valid token can incorporate all earlier valid tokens.

Variable lengths mean the last valid index differs by row. If a Boolean
mask contains `L` true values in a right-padded row, the index is
`L-1`. Gathering with batch indices returns `[B,D]`.


    ### Data-flow walkthrough


For valid lengths `[3,1]`, gather hidden states at positions `[2,0]`.
Compare this with selecting `hidden_states[:,0,:]`, which always uses
position zero regardless of sequence length.


    ### Hints and common mistakes


Reject rows with no valid token. Keep index tensors on the same device
as hidden states. Do not use `hidden_states[:,-1]` because that selects
padding for shorter rows.

In [ ]:
def select_sequence_representation(
    hidden_states: Tensor,
    attention_mask: Tensor,
    position: Literal["first", "last_valid"],
) -> Tensor:
    # LLM_TODO_CH06_03_REPRESENTATION
    raise NotImplementedError("select first or gather last-valid states")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
hidden = torch.arange(2 * 3 * 2).reshape(2, 3, 2)
mask = torch.tensor([[True, True, True], [True, False, False]])
first = select_sequence_representation(hidden, mask, "first")
last = select_sequence_representation(hidden, mask, "last_valid")
assert torch.equal(first, hidden[:, 0])
assert torch.equal(last, torch.stack([hidden[0, 2], hidden[1, 0]]))
print("Exercise 6.3 passed")

## Chapter 06 synthesis


- Which operations need the padding mask besides attention?
- Why must the optimizer be reconsidered after changing trainability?
- How does causal direction affect the information available at position zero?


    After answering these questions, run the chapter's `solution.py`
    from a terminal to compare behavior. The reference file is compact
    by design; the reasoning in this notebook is the main lesson.